In [25]:
import math
from itertools import permutations

In [26]:
def get_user_input():
    try:
        n = int(input("Enter number of equations: "))
        if n < 2:
            print("Number of equations must be >= 2.")
            return None

        print("Enter coefficient matrix A (row-wise):")
        A = []
        for i in range(n):
            row_input = input().split()
            row = []
            for val in row_input:
                row.append(float(val))
            if len(row) != n:
                print("Incorrect number of elements in the row.")
                return None
            A.append(row)

        print("Enter constants vector b:")
        b = []
        for i in range(n):
            b_val = float(input())
            b.append(b_val)

        print("Enter initial guess vector:")
        x0 = []
        for i in range(n):
            x0_val = float(input())
            x0.append(x0_val)

        max_iter = int(input("Enter maximum iterations: "))
        if max_iter <= 0:
            print("Iterations must be greater than 0.")
            return None

        tol = float(input("Enter tolerance: "))
        if tol <= 0:
            print("Tolerance must be greater than 0.")
            return None

        return n, A, b, x0, max_iter, tol
    except ValueError:
        print("Invalid input. Please enter numbers only.")
        return None

In [27]:
def calculate_residual(A, x, b, n):
    residual_sq_sum = 0.0
    for i in range(n):
        ax_i = 0.0
        for j in range(n):
            ax_i = ax_i + A[i][j] * x[j]
        residual_sq_sum = residual_sq_sum + (ax_i - b[i]) ** 2
    return math.sqrt(residual_sq_sum)

In [28]:
def check_diagonal_zeros(A, n):
    for i in range(n):
        if A[i][i] == 0:
            return True
    return False

In [29]:
def rearrange_rows(A, b, n):
    all_orders = permutations(range(n))

    for order in all_orders:

        new_A = []
        new_b = []

        for row_index in order:
            new_A.append(A[row_index])
            new_b.append(b[row_index])

        valid = True

        for i in range(n):
            if new_A[i][i] == 0:
                valid = False
                break

        if valid:
            return new_A, new_b, True

    return A, b, False


def check_diagonal_dominance(A, n):
    for i in range(n):
        diagonal_value = abs(A[i][i])
        non_diagonal_sum = 0.0

        for j in range(n):
            if i != j:
                non_diagonal_sum = non_diagonal_sum + abs(A[i][j])

        if diagonal_value < non_diagonal_sum:
            return False

    return True

In [30]:
def jacobi_method(n, A, b, x0, max_iter, tol):
    print("\n--- Jacobi Method Execution ---")
    x_curr = []
    for val in x0:
        x_curr.append(val)
    
    for k in range(1, max_iter + 1):
        x_new = [0.0] * n
        abs_errors = [0.0] * n
        
        for i in range(n):
            s = 0.0
            for j in range(n):
                if j != i:
                    s = s + A[i][j] * x_curr[j]
            
            x_new[i] = (b[i] - s) / A[i][i]
            abs_errors[i] = abs(x_new[i] - x_curr[i])
            
        residual = calculate_residual(A, x_new, b, n)
        
        print(f"Iteration {k}:")
        
        x_rounded = []
        for val in x_new:
            x_rounded.append(round(val, 6))
        print(f"  x = {x_rounded}")
        
        err_rounded = []
        for err in abs_errors:
            err_rounded.append(round(err, 6))
        print(f"  Absolute Errors = {err_rounded}")
        
        print(f"  Residual Error  = {round(residual, 6)}")
        
        max_err = abs_errors[0]
        for err in abs_errors:
            if err > max_err:
                max_err = err
                
        if max_err <= tol:
            print(f"\nJacobi method converged after {k} iterations.")
            return x_new, k
            
        x_curr = []
        for val in x_new:
            x_curr.append(val)
        
    print(f"\nWarning: Jacobi method failed to converge within {max_iter} iterations.")
    return x_curr, max_iter


In [31]:
def gauss_seidel_method(n, A, b, x0, max_iter, tol):
    print("\n--- Gauss-Seidel Method Execution ---")
    x_curr = []
    for val in x0:
        x_curr.append(val)
    
    for k in range(1, max_iter + 1):
        abs_errors = [0.0] * n
        
        for i in range(n):
            s1 = 0.0
            for j in range(i):
                s1 = s1 + A[i][j] * x_curr[j]
                
            s2 = 0.0
            for j in range(i + 1, n):
                s2 = s2 + A[i][j] * x_curr[j]
            
            new_val = (b[i] - s1 - s2) / A[i][i]
            abs_errors[i] = abs(new_val - x_curr[i])
            x_curr[i] = new_val
            
        residual = calculate_residual(A, x_curr, b, n)
        
        print(f"Iteration {k}:")
        
        x_rounded = []
        for val in x_curr:
            x_rounded.append(round(val, 6))
        print(f"  x = {x_rounded}")
        
        err_rounded = []
        for err in abs_errors:
            err_rounded.append(round(err, 6))
        print(f"  Absolute Errors = {err_rounded}")
        
        print(f"  Residual Error  = {round(residual, 6)}")
        
        max_err = abs_errors[0]
        for err in abs_errors:
            if err > max_err:
                max_err = err
                
        if max_err <= tol:
            print(f"\nGauss-Seidel method converged after {k} iterations.")
            return x_curr, k
            
    print(f"\nWarning: Gauss-Seidel method failed to converge within {max_iter} iterations.")
    return x_curr, max_iter

In [32]:
def main():
    inputs = get_user_input()
    if inputs == None:
        return
        
    n = inputs[0]
    A = inputs[1]
    b = inputs[2]
    x0 = inputs[3]
    max_iter = inputs[4]
    tol = inputs[5]

    if check_diagonal_zeros(A, n) == True:
        print("\nrow swap needed")
        A, b, valid = rearrange_rows(A, b, n)

        if valid:
            print("Rearranged matrix A:")
            for row in A:
                print(row)
            print("Rearranged vector b:")
            print(b)

        else:
            print("Division by zero will occur.")
            return

    if check_diagonal_dominance(A, n) == False:
        print("\nWarning: The coefficient matrix is not diagonally dominant.")
        print("Jacobi and Gauss-Seidel methods may not converge.")
            

    jacobi_sol, jacobi_k = jacobi_method(n, A, b, x0, max_iter, tol)
    gs_sol, gs_k = gauss_seidel_method(n, A, b, x0, max_iter, tol)

    print("\n==================================================")
    print("FINAL RESULTS & COMPARATIVE ANALYSIS")
    print("==================================================")
    
    jacobi_rounded = []
    for x in jacobi_sol:
        jacobi_rounded.append(round(x, 6))
        
    gs_rounded = []
    for x in gs_sol:
        gs_rounded.append(round(x, 6))
    
    print("\n1. Solution Vectors:")
    print(f"   Jacobi Solution       x* = {jacobi_rounded}")
    print(f"   Gauss-Seidel Solution x* = {gs_rounded}")
    
    print("\n2. Convergence Behavior Comparison:")
    print(f"   Jacobi Iterations (k): {jacobi_k}")
    print(f"   Gauss-Seidel Iterations (k): {gs_k}")
    
    print("\n3. Computational Efficiency Remarks:")
    if gs_k < jacobi_k:
        print("   Gauss-Seidel converged faster than the Jacobi method.")
    elif jacobi_k < gs_k:
        print("   Jacobi converged faster than Gauss-Seidel.")
    else:
        print("   Both methods converged in the same number of iterations.")

if __name__ == "__main__":
    main()

Enter coefficient matrix A (row-wise):
Enter constants vector b:
Enter initial guess vector:

--- Jacobi Method Execution ---
Iteration 1:
  x = [0.6, 2.272727, -1.1, 1.875]
  Absolute Errors = [0.6, 2.272727, 1.1, 1.875]
  Residual Error  = 11.353749
Iteration 2:
  x = [1.047273, 1.715909, -0.805227, 0.885227]
  Absolute Errors = [0.447273, 0.556818, 0.294773, 0.989773]
  Residual Error  = 4.990955
Iteration 3:
  x = [0.932636, 2.053306, -1.049341, 1.130881]
  Absolute Errors = [0.114636, 0.337397, 0.244114, 0.245653]
  Residual Error  = 2.029878
Iteration 4:
  x = [1.015199, 1.953696, -0.968109, 0.973843]
  Absolute Errors = [0.082562, 0.09961, 0.081232, 0.157038]
  Residual Error  = 0.891141
Iteration 5:
  x = [0.988991, 2.011415, -1.010286, 1.021351]
  Absolute Errors = [0.026207, 0.057719, 0.042177, 0.047508]
  Residual Error  = 0.368628
Iteration 6:
  x = [1.003199, 1.992241, -0.994522, 0.994434]
  Absolute Errors = [0.014207, 0.019173, 0.015764, 0.026917]
  Residual Error  = 0.1